# Evaluate UNet Segmentation with Skeleton Metrics

Runs `segmentation_skeleton_metrics.evaluate.evaluate(...)` on the same brain/segmentation pair used by `load_skeletons.ipynb`. Outputs a CSV report and a text summary in `metrics_out/`.

**Note:** This re-reads the SWCs from GCS (~15 min) because the metrics package builds its own labeled-graph representation that differs from `BrainDataset`. The `dataset_cache_*.pkl` does not help here.

### Imports

In [1]:
import os

from segmentation_skeleton_metrics.evaluate import evaluate
from segmentation_skeleton_metrics.utils.img_util import TensorStoreImage

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

### Paths

In [2]:
# Match load_skeletons.ipynb so we evaluate the same UNet checkpoint.
brain_id = "794495"
segmentation_id = "raw.unet_449_splits_and_merges_900000"

gt_path = f"gs://allen-nd-goog/ground_truth_tracings/{brain_id}/voxel"
fragments_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/swcs"
segmentation_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/"

output_dir = f"metrics_out/{brain_id}/{segmentation_id}"
os.makedirs(output_dir, exist_ok=True)

### Run Evaluation

Three-step pipeline run by `evaluate(...)`:
1. **Label graphs** -- read GT SWCs and label each node with the UNet segment ID it falls inside (uses `segmentation`).
2. **Detect errors** -- per edge, classify as correct / split / omit / merge by comparing endpoint labels.
3. **Compute metrics** -- # Splits, # Merges, % Split/Omit/Merged Edges, ERL, Normalized ERL, Edge Accuracy, Split Rate, Merge Rate.

Saved artifacts: `results.csv` (per-GT-SWC), `results_overview.txt` (averaged + totals).

In [3]:
# anisotropy: same (x, y, z) microns/voxel as load_skeletons.ipynb. Not
#   applied to the GT SWCs because gt_path ends in /voxel (already in voxel
#   coords); use_anisotropy=False reflects that.
anisotropy = (0.748, 0.748, 1.0)

# swap_axes=True: SWC voxel coords are stored (x, y, z); after the
# `from_google` transpose the segmentation volume's last 3 dims are
# (z, y, x). Setting swap_axes adds an additional transpose so the image's
# axes line up with the SWC's (x, y, z) ordering. Without this you get
# IndexError: OUT_OF_RANGE on the first GT skeleton labeling pass.
segmentation = TensorStoreImage(segmentation_path, swap_axes=True)

evaluate(
    gt_path,
    segmentation,
    output_dir,
    anisotropy=anisotropy,
    fragments_path=fragments_path,
    use_anisotropy=False,
    save_merges=True,
    save_fragments=False,
    verbose=True,
)

I0524 15:07:36.398538 3473968 google_auth_provider.cc:149] Using credentials at zihan_gcs_token.json
I0524 15:07:36.399434 3473968 google_auth_provider.cc:165] Using ServiceAccount AuthProvider



(1) Load Ground Truth


Label Graphs: 100%|██████████| 19/19 [03:27<00:00, 10.93s/it]



(2) Load Fragments


Build Graphs: 100%|██████████| 484546/484546 [13:21<00:00, 604.68it/s] 



(3) Compute Metrics


Added Cable Length (μm): 100%|██████████| 124/124 [06:09<00:00,  2.98s/it]



Average Results...
  # Splits: 571.3671
  # Merges: 6.9252
  % Split Edges: 0.3154
  % Omit Edges: 1.3572
  % Merged Edges: 24.5452
  ERL: 5090.5640
  Normalized ERL: 0.0095
  Edge Accuracy: 73.7819
  Split Rate: 958.7202
  Merge Rate: 100277.8758

Total Results...
  # Splits: 10031
  # Merges: 124


### Inspect Results

In [4]:
import pandas as pd

results = pd.read_csv(os.path.join(output_dir, "results.csv"), index_col=0)
print(f"Per-SWC results ({len(results)} ground-truth skeletons):")
results

Per-SWC results (19 ground-truth skeletons):


,SWC Run Length,# Splits,# Merges,% Split Edges,% Omit Edges,% Merged Edges,ERL,Normalized ERL,Edge Accuracy,Split Rate,Merge Rate
N001-794495-JT,348316.058568,278.0,4,0.238656,0.868863,24.608352,4666.27,0.0134,74.28,1239.06,86114.60
N002-794495-PP,297708.485078,456.0,7,0.408412,2.261733,9.075177,2183.21,0.0073,88.25,635.44,41394.18
N003-794495-SP,344438.058303,1044.0,12,0.870149,2.831723,18.639362,1646.76,0.0048,77.66,317.71,27640.62
N004-794495-JG,424002.004813,797.0,4,0.453777,2.891457,28.076224,2207.42,0.0052,68.58,514.20,102454.54
N005-794495-HP,436906.054243,542.0,2,0.297413,1.671425,23.735765,3048.33,0.0070,74.30,790.23,214152.04
N006-794495-PP,443186.352157,799.0,4,0.565378,1.286940,13.857762,2951.03,0.0067,84.29,544.40,108744.28
N007-794495-JG,712366.889009,774.0,8,0.288301,1.285609,24.780980,7139.03,0.0100,73.65,905.88,87644.36
N008-794495-SP,285232.039222,285.0,3,0.262017,0.969990,21.525160,4170.11,0.0146,77.24,988.48,93905.99
N009-794495-MB,606058.496239,811.0,7,0.338479,0.923563,19.809793,9096.00,0.0150,78.93,737.87,85487.11
N011-794495-PP,400228.062377,628.0,27,0.393360,1.672357,28.319902,3942.88,0.0099,69.61,624.14,14517.06


In [5]:
with open(os.path.join(output_dir, "results_overview.txt")) as f:
    print(f.read())


Average Results...
  # Splits: 571.3671
  # Merges: 6.9252
  % Split Edges: 0.3154
  % Omit Edges: 1.3572
  % Merged Edges: 24.5452
  ERL: 5090.5640
  Normalized ERL: 0.0095
  Edge Accuracy: 73.7819
  Split Rate: 958.7202
  Merge Rate: 100277.8758

Total Results...
  # Splits: 10031
  # Merges: 124

